# Preprocesamiento con CKDPreprocessor

Aplica el pipeline de preprocesamiento y verifica que no queden valores faltantes.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # raiz del proyecto para importar src
import logging
logging.basicConfig(level=logging.INFO)

In [2]:
from src import data_loader as dl, preprocessing as pp
df = dl.load_ckd_dataset()
X_train, X_test, y_train, y_test = dl.split_data(df)
print('train:', X_train.shape, '| test:', X_test.shape)

INFO:src.data_loader:Dataset crudo cargado: 400 filas x 25 columnas


INFO:src.data_loader:Particion estratificada: train=320, test=80 (positivos train=0.62, test=0.62)


train: (320, 24) | test: (80, 24)


## Antes: valores faltantes en el conjunto de entrenamiento

In [3]:
X_train.isna().sum().sort_values(ascending=False).head(10)

rbc     126
rc      105
wc       84
pot      76
sod      75
pcv      58
pc       54
hemo     43
su       42
sg       41
dtype: int64

## Ajuste del preprocesador (solo con train) y transformación

In [4]:
prep = pp.CKDPreprocessor()
X_train_p = prep.fit_transform(X_train)
X_test_p = prep.transform(X_test)
print('features de salida:', len(prep.feature_names_))
X_train_p.head()

INFO:src.preprocessing:CKDPreprocessor ajustado: 28 features de salida


features de salida: 28


,age,bp,sg,al,su,bgr,bu,sc,sod,pot,...,pe,ane,rbc_abnormal,rbc_normal,pc_abnormal,pc_normal,pcc_notpresent,pcc_present,ba_notpresent,ba_present
108,0.488636,0.230769,0.50,0.00,0.00,0.181624,0.042122,0.007937,0.938144,0.333333,...,0,0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
210,0.647727,0.384615,0.50,0.80,0.40,0.497863,0.407176,0.164021,0.896907,0.627451,...,0,1,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
137,0.488636,0.076923,0.25,0.40,0.00,0.525641,0.263651,0.047619,0.890034,0.509804,...,0,0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0
148,0.761364,0.076923,0.75,0.12,0.08,0.318376,0.076443,0.630952,0.927148,0.309804,...,0,0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
246,0.522727,0.461538,0.50,0.60,0.00,0.179487,0.666147,0.195767,0.793814,0.627451,...,0,1,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


## Después: verificación de que no hay NaN y valores en [0, 1]

In [5]:
print('NaN en train:', int(X_train_p.isna().sum().sum()))
print('NaN en test :', int(X_test_p.isna().sum().sum()))
print('rango:', float(X_train_p.values.min()), '-', float(X_train_p.values.max()))

NaN en train: 0
NaN en test : 0
rango: 0.0 - 1.0000000000000002


## SMOTE (solo sobre el entrenamiento)

In [6]:
X_res, y_res = pp.apply_smote(X_train_p, y_train)
print('antes:', dict(y_train.value_counts()))
print('después:', dict(y_res.value_counts()))

INFO:src.preprocessing:SMOTE aplicado: 320 -> 400 filas (positivos 0.62 -> 0.50)


antes: {1: 200, 0: 120}
después: {1: 200, 0: 200}
